In [82]:
import os
import io
import sys
from dotenv import load_dotenv
from groq import Groq
from openai import OpenAI
import gradio as gr
import subprocess

In [83]:
load_dotenv(override=True)
groq_api_key = os.getenv('GROQ_API_KEY')


if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")


Groq API Key exists and begins gsk_


In [84]:
groq=Groq()
openai=OpenAI()
ollama_url ="http://localhost:11434/v1"
groq = Groq(api_key=groq_api_key)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [85]:
models = [ "qwen2.5-coder:3b", "openai/gpt-oss-120b", "llama-3.3-70b-versatile"]

clients = { "openai/gpt-oss-120b": groq, "qwen2.5-coder:3b": ollama,"llama-3.3-70b-versatile":groq}

In [49]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Windows',
  'arch': 'AMD64',
  'release': '11',
  'version': '10.0.26200',
  'kernel': '11',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'mingw32'},
 'package_managers': ['winget'],
 'cpu': {'brand': 'Intel(R) Core(TM) i5-8350U CPU @ 1.70GHz',
  'cores_logical': 8,
  'cores_physical': 4,
  'simd': []},
 'toolchain': {'compilers': {'gcc': 'gcc.exe (MinGW.org GCC-6.3.0-1) 6.3.0',
   'g++': 'g++.exe (MinGW.org GCC-6.3.0-1) 6.3.0',
   'clang': '',
   'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': ''},
  'linkers': {'ld_lld': ''}}}

In [86]:
compile_command = [
    "g++",
    "-std=c++17",
    "-O3",
    "main.cpp",
    "-o",
    "main.exe"
]

run_command = [r".\main.exe"]

In [87]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [88]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]

In [89]:
def write_output(cpp):
    with open("main.cpp", "w") as f:
        f.write(cpp)

In [90]:
def port(model, python):
    client = clients[model]

    response = client.chat.completions.create(
        model=model,
        messages=messages_for(python)
    )

    reply = response.choices[0].message.content

    
    reply = reply.replace("```cpp", "").replace("```", "").strip()

    write_output(reply)
    return reply

In [92]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200000000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [93]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [98]:
run_python(pi)

'Result: 3.141592656089\nExecution Time: 44.605016 seconds\n'

In [94]:
def compile_and_run():
    print("Compiling...")

    compile_result = subprocess.run(
        compile_command,
        text=True,
        capture_output=True
    )

    if compile_result.returncode != 0:
        print("Compilation Failed")
        print(compile_result.stderr)
        return

    print("Compilation Successful")

    exe_path = os.path.abspath("main.exe")

    if not os.path.exists(exe_path):
        print(f" Executable not found: {exe_path}")
        return

    print(f"Running: {exe_path}")

    run_result = subprocess.run(
        exe_path,
        text=True,
        capture_output=True,
        shell=True
    )

    print("Return Code:", run_result.returncode)

    if run_result.stdout:
        print("STDOUT:")
        print(run_result.stdout)

    if run_result.stderr:
        print("STDERR:")
        print(run_result.stderr)

In [67]:
print(compile_command)
print(run_command)

['clang++', '-std=c++17', '-O3', 'main.cpp', '-o', 'main.exe']
['.\\main.exe']


In [95]:
with gr.Blocks() as ui:
    with gr.Row():
        python = gr.Textbox(label="Python code:", lines=28, value=pi)
        cpp = gr.Textbox(label="C++ code:", lines=28)
    with gr.Row():
        model = gr.Dropdown(models, label="Select model", value=models[0])
        convert = gr.Button("Convert code")

    convert.click(port, inputs=[model, python], outputs=[cpp])

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


In [97]:
compile_and_run()

Compiling...
Compilation Successful
Running: c:\Users\Rana\projectt\llm\LLMShowdown\main.exe
Return Code: 0
STDOUT:
Result: 3.141592656090
Execution Time: 0.736488 seconds



In [99]:
44.605016/0.736488

60.56448441794028

gpt-oss-120b=0.736488
qwen-2.5-coder-3b=failed    

gpt-oss-120 is 60.56 times faster



